# Anarkali scenario test notebook

This notebook tests the model on realistic decision scenarios, not just a single happy-path example.
It checks whether the model picks the right route, probability shape, and confidence behaviour.

In [1]:
from anarkali import Engine

engine = Engine.load("release-150m")
print("Loaded model:", engine.name)

2026-10-03 09:07:07.851426496 [W:onnxruntime:Default, device_discovery.cc:146 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


Loaded model: Anarkali 150M


In [4]:
scenarios = {
    "support_routing": {
        "state": {
            "customer": "A customer reports they cannot access the portal after password reset.",
            "context": ["MFA required", "user can login to mobile but not desktop"],
            "severity": "medium"
        },
        "questions": {
            "route": {
                "type": "choice",
                "instructions": "Which route best matches this issue?",
                "criteria": {
                    "identity": "Identity or account access problem",
                    "security": "Security incident or breach",
                    "product": "Product bug or feature defect",
                    "billing": "Billing or payment issue"
                }
            }
        }
    },
    "ci_failure": {
        "state": {
            "repo": "payments-service",
            "error": "pytest fails on invoice export job",
            "logs": ["KeyError: 'currency'", "job started after deploy"]
        },
        "questions": {
            "cause": {
                "type": "choice",
                "instructions": "What is the most likely cause of the CI failure?",
                "criteria": {
                    "schema_change": "Recent schema or contract change",
                    "environment": "Environment or dependency issue",
                    "test_flake": "Test flake or nondeterministic failure",
                    "infra": "Infrastructure or runner issue"
                }
            }
        }
    },
    "invoice_approval": {
        "state": {
            "invoice": "INV-2026-0917",
            "amount": 48120,
            "currency": "INR",
            "approval_policy": "Invoices above 40000 INR require finance approval before payment."
        },
        "questions": {
            "needs_finance_approval": {
                "type": "noul",
                "instructions": "The invoice needs finance approval before payment.",
                "criteria": {
                    "false": "The invoice can be paid without finance approval.",
                    "true": "The invoice must be approved by finance before payment."
                }
            }
        }
    },
    "risk_level": {
        "state": {
            "incident": "API latency spiked for 10 minutes across 2 regions.",
            "evidence": ["customer complaints", "partial outage", "no confirmed breach"]
        },
        "questions": {
            "risk": {
                "type": "score",
                "instructions": "Estimate the operational risk level.",
                "criteria": ["low", "medium", "high", "critical"]
            }
        }
    },
    "product_vs_incident": {
        "state": {
            "description": "A dozen users report missing export button after the latest release.",
            "activity": ["single feature affected", "no signs of broader outage"]
        },
        "questions": {
            "classification": {
                "type": "choice",
                "instructions": "How should this issue be classified?",
                "criteria": {
                    "product": "Product bug or regression",
                    "incident": "Widespread service incident",
                    "security": "Security problem",
                    "billing": "Billing problem"
                }
            }
        }
    },
    "suspicious_login": {
        "state": {
            "customer": "User reports login from a new device in Singapore despite being in Mumbai.",
            "signals": ["same session token used", "MFA passed", "no malware report"],
            "risk": "medium"
        },
        "questions": {
            "route": {
                "type": "choice",
                "instructions": "Which handling path is best?",
                "criteria": {
                    "identity": "Account identity verification / suspicious login",
                    "security": "Active security incident or breach",
                    "product": "Product defect or feature issue",
                    "billing": "Billing or payment problem"
                }
            }
        }
    },
    "refund_request": {
        "state": {
            "customer": "Customer wants refund after duplicate charge on monthly subscription.",
            "amount": 299,
            "status": "paid",
            "history": ["first charge succeeded", "second charge appears across 2 subscriptions"]
        },
        "questions": {
            "manual_review_needed": {
                "type": "noul",
                "instructions": "The refund request likely requires manual finance review.",
                "criteria": {
                    "false": "The case can be handled automatically.",
                    "true": "The case should be escalated for manual review."
                }
            }
        }
    },
    "platform_outage": {
        "state": {
            "incident": "Checkout API latency increased by 9x for 20 minutes and 30% of checkouts failed.",
            "signals": ["customer impact across 3 regions", "no confirmed breach", "recent deploy happened 40 minutes ago"],
            "owner": "platform"
        },
        "questions": {
            "risk": {
                "type": "score",
                "instructions": "Rate the severity of the operational incident.",
                "criteria": ["low", "medium", "high", "critical"]
            }
        }
    },
    "data_exposure": {
        "state": {
            "description": "Support ticket says a third-party script accessed user export data without consent.",
            "details": ["export logs show 4k records", "customer reported no unauthorized login"],
            "sensitivity": "high"
        },
        "questions": {
            "classification": {
                "type": "choice",
                "instructions": "How should this problem be classified?",
                "criteria": {
                    "security": "Security or privacy incident",
                    "product": "Product bug or feature issue",
                    "identity": "Identity or access issue",
                    "billing": "Billing or payment issue"
                }
            }
        }
    }
}

for name, payload in scenarios.items():
    print(f"\n=== {name} ===")
    result = engine.predict(payload["state"], payload["questions"])
    print(result)



=== support_routing ===
{'model': 'Anarkali 150M', 'answers': {'route': {'type': 'choice', 'choice': 'identity', 'probabilities': {'identity': 0.3377, 'security': 0.2796, 'product': 0.2414, 'billing': 0.1413}, 'confidence': 0.1169}}, 'usage': {'input_tokens': 69, 'output_tokens': 0}, 'latency_ms': 533.97}

=== ci_failure ===
{'model': 'Anarkali 150M', 'answers': {'cause': {'type': 'choice', 'choice': 'test_flake', 'probabilities': {'schema_change': 0.1309, 'environment': 0.2502, 'test_flake': 0.3114, 'infra': 0.3075}, 'confidence': 0.0818}}, 'usage': {'input_tokens': 75, 'output_tokens': 0}, 'latency_ms': 405.83}

=== invoice_approval ===
{'model': 'Anarkali 150M', 'answers': {'needs_finance_approval': {'type': 'noul', 'noul': 0.7832, 'confidence': 0.7832}}, 'usage': {'input_tokens': 78, 'output_tokens': 0}, 'latency_ms': 355.53}

=== risk_level ===
{'model': 'Anarkali 150M', 'answers': {'risk': {'type': 'score', 'score': 1.336, 'legend': {'0': 'low', '1': 'medium', '2': 'high', '3': 

In [5]:
def summarize_answer(answer):
    kind = answer["type"]
    if kind == "choice":
        return {
            "picked": answer["choice"],
            "confidence": answer["confidence"],
            "top_prob": max(answer["probabilities"].values()),
            "type": kind,
        }
    if kind == "noul":
        return {
            "prob_true": answer["noul"],
            "confidence": answer["confidence"],
            "type": kind,
        }
    return {
        "score": answer["score"],
        "confidence": answer["confidence"],
        "type": kind,
    }

for name, payload in scenarios.items():
    result = engine.predict(payload["state"], payload["questions"])
    summary = {qid: summarize_answer(answer) for qid, answer in result["answers"].items()}
    print(f"\n{name}: {summary}")


support_routing: {'route': {'picked': 'identity', 'confidence': 0.1169, 'top_prob': 0.3377, 'type': 'choice'}}

ci_failure: {'cause': {'picked': 'test_flake', 'confidence': 0.0818, 'top_prob': 0.3114, 'type': 'choice'}}

invoice_approval: {'needs_finance_approval': {'prob_true': 0.7832, 'confidence': 0.7832, 'type': 'noul'}}

risk_level: {'risk': {'score': 1.336, 'confidence': 0.0535, 'type': 'score'}}

product_vs_incident: {'classification': {'picked': 'product', 'confidence': 0.0657, 'top_prob': 0.2993, 'type': 'choice'}}

suspicious_login: {'route': {'picked': 'identity', 'confidence': 0.209, 'top_prob': 0.4067, 'type': 'choice'}}

refund_request: {'manual_review_needed': {'prob_true': 0.7046, 'confidence': 0.7046, 'type': 'noul'}}

platform_outage: {'risk': {'score': 1.8434, 'confidence': 0.1067, 'type': 'score'}}

data_exposure: {'classification': {'picked': 'identity', 'confidence': 0.0757, 'top_prob': 0.3068, 'type': 'choice'}}


## How to read the result

- `choice`: model picks one option and gives a probability for each option.
- `noul`: model gives a probability that the statement is true.
- `score`: model gives a level estimate on an ordered scale.
- `confidence`: a rough measure of certainty, but still should be treated as advisory.

If the model is uncertain, route to a human reviewer rather than acting automatically.
This is especially important for CI, security, and billing workflows.